# ISLES'24 Challenge: Interactive Clinical & Lesion Data Explorer
This notebook provides an interactive dashboard to explore clinical baseline characteristics, workflow times, and ischemic lesion pattern metrics from the **ISLES'24 dataset**.

### Features:
- **Dynamic Stratification**: Compare features across **All Data**, **Train vs. Test**, or **Center 1 vs. Center 2**.
- **Interactive Visualizations**: Powered by **Plotly** and **ipywidgets** (Boxplots, Violin distributions, Grouped Bar charts).
- **Automated Hypothesis Testing**: Computes non-parametric **Mann-Whitney U** tests for continuous variables and **Fisher's Exact / Chi-Square** tests for categorical features.


In [ ]:
# Setup & Imports
import datetime
import ipywidgets as widgets
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import clear_output, display
from scipy import stats

# Path configurations
PATH_TRAIN_PRE = '/home/edelarosa/Documents/git/multimodal_lofo_isles24/ISLES24-Multimodal/data/data/all_clinical_data-media/all_train_data-pre_FINAL_MEDIA.xlsx'
PATH_TRAIN_POST = '/home/edelarosa/Documents/git/multimodal_lofo_isles24/ISLES24-Multimodal/data/data/all_clinical_data-media/all_train_data-post_FINAL_MEDIA.xlsx'
PATH_TEST_PRE = '/home/edelarosa/Documents/git/multimodal_lofo_isles24/ISLES24-Multimodal/data/data/all_clinical_data-media/all_test_data-pre.xlsx'
PATH_TEST_POST = '/home/edelarosa/Documents/git/multimodal_lofo_isles24/ISLES24-Multimodal/data/data/all_clinical_data-media/all_test_data-post.xlsx'
PATH_ISLES_STATS = '/home/edelarosa/Documents/git/multimodal_lofo_isles24/ISLES24-Multimodal/data/data/all_clinical_data-media/isles24_stats_forMedIA_final.xlsx'


In [ ]:
# 1. Load & Preprocess Data
def load_isles_data():
    train_pre = pd.read_excel(PATH_TRAIN_PRE).drop_duplicates(subset=['patient_id'])
    train_post = pd.read_excel(PATH_TRAIN_POST).drop_duplicates(subset=['patient_id'])
    test_pre = pd.read_excel(PATH_TEST_PRE).drop_duplicates(subset=['patient_id'])
    test_post = pd.read_excel(PATH_TEST_POST).drop_duplicates(subset=['patient_id'])
    isles_stats = pd.read_excel(PATH_ISLES_STATS)
    
    test_post['TICI postinterventional'] = test_post['TICI postinterventional'].fillna(test_post.get('Tici postinterventional', pd.Series(dtype=object)))
    test_post['mRS discharge'] = test_post['mRS discharge'].fillna(test_post.get('MRS discharge', pd.Series(dtype=object)))
    test_post['mRS 3 months'] = test_post['mRS 3 months'].fillna(test_post.get('MRS 3 months', pd.Series(dtype=object)))
    test_post = test_post.drop(columns=['Tici postinterventional', 'MRS discharge', 'MRS 3 months'], errors='ignore')
    
    train_merged = pd.merge(train_pre, train_post, on='patient_id', how='outer')
    train_merged['Subset'] = 'Train'
    test_merged = pd.merge(test_pre, test_post, on='patient_id', how='outer')
    test_merged['Subset'] = 'Test'
    
    df_clinical = pd.concat([train_merged, test_merged], ignore_index=True)
    
    common_cols = [c for c in df_clinical.columns if c in isles_stats.columns and c not in ['patient_id', 'isles_id', 'Subset']]
    for col in common_cols:
        isles_map = isles_stats.set_index('isles_id')[col].dropna().to_dict()
        df_clinical[col] = df_clinical['patient_id'].map(isles_map).fillna(df_clinical[col])
    
    df_clinical['Lypid lowering drugs combined'] = df_clinical['Lipid lowering drugs'].fillna(df_clinical.get('Statins', pd.Series(dtype=object)))
    
    if 'infarcted_structure' in isles_stats.columns:
        isles_stats['infarcted_structure'] = isles_stats['infarcted_structure'].fillna('None')
        
    return df_clinical, isles_stats

df_clinical, isles_stats = load_isles_data()
print(f'Loaded Clinical Data: {len(df_clinical)} subjects | Lesion Stats: {len(isles_stats)} subjects')


In [ ]:
# 2. Helper Functions & Time Parsers
def parse_time_to_minutes(val):
    if pd.isna(val) or val is None:
        return np.nan
    if isinstance(val, (datetime.time, pd.Timestamp, datetime.datetime)):
        return val.hour * 60.0 + val.minute + val.second + val.microsecond / 1e6
    val_str = str(val).strip()
    if val_str == '' or val_str.lower() in ['nan', 'none', 'nat']:
        return np.nan
    if ' ' in val_str:
        val_str = val_str.split(' ')[-1]
    try:
        parts = val_str.split(':')
        if len(parts) == 3:
            return float(parts[0]) * 60.0 + float(parts[1]) + float(parts[2]) / 60.0
        elif len(parts) == 2:
            return float(parts[0]) + float(parts[1]) / 60.0
        else:
            return float(val_str)
    except Exception:
        return np.nan

VAR_CONFIGS = {
    'Age (years)': {'df': 'clinical', 'col': 'Age', 'type': 'continuous'},
    'Sex': {'df': 'clinical', 'col': 'Sex', 'type': 'categorical'},
    'Center': {'df': 'clinical', 'col': 'Center', 'type': 'categorical'},
    'Atrial Fibrillation': {'df': 'clinical', 'col': 'Atrial fibrillation', 'type': 'categorical'},
    'Hypertension': {'df': 'clinical', 'col': 'Hypertension', 'type': 'categorical'},
    'Diabetes': {'df': 'clinical', 'col': 'Diabetes', 'type': 'categorical'},
    'Hyperlipidemia': {'df': 'clinical', 'col': 'Hyperlipidemia', 'type': 'categorical'},
    'Glucose (mg/dL)': {'df': 'clinical', 'col': 'Glucose', 'type': 'continuous'},
    'NIHSS at Admission': {'df': 'clinical', 'col': 'NIHSS at admission', 'type': 'continuous'},
    'mRS at Admission': {'df': 'clinical', 'col': 'mRS at admission', 'type': 'continuous'},
    'mRS 3 Months': {'df': 'clinical', 'col': 'mRS 3 months', 'type': 'continuous'},
    'Onset to Door Time (min)': {'df': 'clinical', 'col': 'Onset to door', 'type': 'time'},
    'Door to Imaging Time (min)': {'df': 'clinical', 'col': 'Door to imaging', 'type': 'time'},
    'Door to Groin Time (min)': {'df': 'clinical', 'col': 'Door to groin', 'type': 'time'},
    'Door to Recanalization Time (min)': {'df': 'clinical', 'col': 'Door to recanalization', 'type': 'time'},
    'Total Lesion Volume (mL)': {'df': 'lesion', 'col': 'total_lesion_volume', 'type': 'continuous'},
    'Number of Lesions': {'df': 'lesion', 'col': 'number_lesions', 'type': 'continuous'},
    'Max Lesion Vol / Total Ratio': {'df': 'lesion', 'col': 'max_lesion_vol/total_lesion_volume', 'type': 'continuous'},
    'Infarct Pattern Group': {'df': 'lesion', 'col': 'group', 'type': 'categorical'},
    'Infarcted Brain Structure': {'df': 'lesion', 'col': 'infarcted_structure', 'type': 'categorical'},
}


In [ ]:
# 3. Interactive Plotting Dashboard
def plot_interactive_variable(variable_name, stratification):
    cfg = VAR_CONFIGS[variable_name]
    df_src = df_clinical.copy() if cfg['df'] == 'clinical' else isles_stats.copy()
    col = cfg['col']
    vtype = cfg['type']
    
    if vtype == 'time':
        df_src[col] = df_src[col].apply(parse_time_to_minutes)
        vtype = 'continuous'
    
    if stratification == 'All Data':
        df_src['Strata'] = 'All Patients (N=248)'
        group_col = 'Strata'
    elif stratification == 'Train vs Test':
        group_col = 'Subset'
    elif stratification == 'Center 1 vs Center 2':
        df_src['Center_Strata'] = df_src['Center'].map({1: 'Center 1', 2: 'Center 2'})
        group_col = 'Center_Strata'
    
    p_value_str = '-'
    test_name = ''
    if stratification != 'All Data':
        groups = df_src[group_col].dropna().unique()
        if len(groups) == 2:
            g1 = df_src[df_src[group_col] == groups[0]][col].dropna()
            g2 = df_src[df_src[group_col] == groups[1]][col].dropna()
            if vtype == 'continuous':
                stat, p_val = stats.mannwhitneyu(pd.to_numeric(g1, errors='coerce').dropna(),
                                                pd.to_numeric(g2, errors='coerce').dropna())
                test_name = 'Mann-Whitney U test'
            else:
                ctab = pd.crosstab(df_src[group_col], df_src[col])
                if ctab.shape == (2, 2):
                    _, p_val = stats.fisher_exact(ctab)
                    test_name = 'Fisher\'s Exact test'
                else:
                    _, p_val, _, _ = stats.chi2_contingency(ctab)
                    test_name = 'Chi-Square test'
            p_value_str = '< 0.001' if p_val < 0.001 else f'{p_val:.4f}'
            
    total_missing = df_src[col].isna().sum()
    print(f'=== {variable_name} ({stratification}) ===')
    print(f'Total valid entries: {len(df_src[col].dropna())} | Missing cases: {total_missing}')
    if stratification != 'All Data':
        print(f'Statistical Comparison ({test_name}): p-value = {p_value_str}')
    print('-' * 60)
    
    if vtype == 'continuous':
        fig = px.box(
            df_src.dropna(subset=[col]),
            x=group_col,
            y=col,
            color=group_col,
            points='all',
            boxmode='overlay',
            title=f'{variable_name} Distribution ({stratification}) - p: {p_value_str}',
            labels={col: variable_name, group_col: 'Stratification Group'},
            template='plotly_white',
            color_discrete_sequence=px.colors.qualitative.Prism
        )
        fig.update_traces(jitter=0.35, pointpos=-1.8, marker=dict(size=5, opacity=0.6))
    else:
        df_counts = df_src.groupby([group_col, col]).size().reset_index(name='Count')
        df_counts['Percentage'] = df_counts.groupby(group_col)['Count'].transform(lambda x: x / x.sum() * 100)
        
        fig = px.bar(
            df_counts,
            x=col,
            y='Count',
            color=group_col,
            barmode='group',
            text=df_counts['Percentage'].apply(lambda x: f'{x:.1f}%'),
            title=f'{variable_name} Distribution ({stratification}) - p: {p_value_str}',
            labels={col: variable_name, 'Count': 'Patient Count'},
            template='plotly_white',
            color_discrete_sequence=px.colors.qualitative.Safe
        )
        fig.update_traces(textposition='outside')
        
    fig.update_layout(
        height=520,
        font=dict(family='Calibri', size=13),
        showlegend=True,
        title_x=0.5
    )
    fig.show()

var_dropdown = widgets.Dropdown(
    options=list(VAR_CONFIGS.keys()),
    value='Total Lesion Volume (mL)',
    description='Variable:',
    style={'description_width': 'initial'}
)

strata_dropdown = widgets.Dropdown(
    options=['All Data', 'Train vs Test', 'Center 1 vs Center 2'],
    value='Train vs Test',
    description='Stratification:',
    style={'description_width': 'initial'}
)

dashboard = widgets.interactive(plot_interactive_variable, variable_name=var_dropdown, stratification=strata_dropdown)
display(dashboard)
